# Etapa 1 — Validar persistência Delta
Execute todas as células no Databricks. O notebook cria apenas o schema de diagnóstico `medallion_dev_setup` no catálogo informado e a tabela `teste_delta`. Não remove objetos.
Valida criação de tabela, MERGE, atualização, releitura e repetição sem duplicidade. Isto ainda não comprova a idempotência do pipeline completo.
Ao concluir, compartilhe a saída da última célula. Não compartilhe credenciais.


In [0]:
import re

dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())
catalogo = dbutils.widgets.get("catalogo").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo):
    raise ValueError("Informe um catálogo com letras, números e sublinhado.")

schema = f"`{catalogo}`.`medallion_dev_setup`"
tabela = f"{schema}.`teste_delta`"
print(f"Spark: {spark.version}")
print(f"Destino do teste: {tabela}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabela} (
        id BIGINT,
        descricao STRING
    ) USING DELTA
""")
formato = spark.sql(f"DESCRIBE DETAIL {tabela}").first()["format"]
assert formato.lower() == "delta", f"Formato inesperado: {formato}"



In [0]:
def gravar(descricao):
    # Valores literais controlados exclusivamente por este notebook.
    if descricao not in ("inserido", "atualizado"):
        raise ValueError("Descrição de teste inválida")
    spark.sql(f"""
        MERGE INTO {tabela} AS destino
        USING (SELECT CAST(1 AS BIGINT) AS id, '{descricao}' AS descricao) AS origem
        ON destino.id = origem.id
        WHEN MATCHED THEN UPDATE SET descricao = origem.descricao
        WHEN NOT MATCHED THEN INSERT (id, descricao)
        VALUES (origem.id, origem.descricao)
    """)

gravar("inserido")
gravar("atualizado")
gravar("atualizado")

resultado = spark.table(tabela).where("id = 1").collect()
assert len(resultado) == 1, f"Esperada uma linha; encontradas {len(resultado)}"
assert resultado[0]["descricao"] == "atualizado", "Atualização não persistida"

print("VALIDAÇÃO DELTA: OK")
print(f"Catálogo: {catalogo}")
print("Schema: medallion_dev_setup")
print("Formato: delta")
print("MERGE e releitura: OK")
print("Repetição sem duplicidade para id=1: OK")

